# LCDM CMB + BAO + BBN MCMC Driver

This notebook runs **one** posterior combination selected by `ACTIVE_COMBINATION` and saves the posterior artifact to disk.
Run it once per combination, then use the plotting notebook to aggregate the saved outputs.


In [1]:
import os
from pathlib import Path

os.environ.setdefault('XDG_CACHE_HOME', str(Path.cwd() / '.cache'))
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.cache' / 'matplotlib'))
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

import jax
jax.config.update('jax_enable_x64', True)

import jax.numpy as jnp
import numpy as np
import pandas as pd
from IPython.display import display
from numpy.linalg import inv

import candl_data
from ps_1loop_jax import background as bg

from jaxptpolypol.bao import load_desi_dr2, make_bao_theory_fn
from jaxptpolypol.cmb import (
    CandlParameterLayout,
    get_candl_default_parameters,
    get_candl_parameter_names,
    load_candl_likelihood,
    make_candl_loglike_fn,
    make_candl_pars_to_theory_specs_fn,
)
from jaxptpolypol.cmb_mcmc_utils import (
    COMBINATION_CONFIGS,
    COSMO_KEYS_NATIVE,
    COSMO_SIZES_NATIVE,
    DEFAULT_FIDUCIAL_NATIVE,
    artifact_path_for_selector,
    get_cache_dir,
    native_cosmo_dict_from_sampled,
    ordered_union,
    save_run_artifact,
    scalar_value,
)
from jaxptpolypol.params import CosmoParams
from jaxptpolypol.sampler import make_transform, run_nuts, samples_to_physical


In [2]:
REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
for _base in REPO_ROOT_CANDIDATES:
    if (_base / 'ext_data/bao_data/desi_bao_dr2').exists():
        REPO_ROOT = _base
        break
else:
    REPO_ROOT = Path.cwd()

COMBINATIONS = ["bao_bbn",
                "act_lensing_bbn",
                "act_lensing_bao",
                "act_planck_lensing_bao",
                "planck_lensing_bao",
                "planck_primary",
               ]
ACTIVE_COMBINATION = COMBINATIONS[5]
SEED = 43
MNU_FIXED = 0.06
NEFF_FIXED = 3.046
MARGINALIZE_CMB_NUISANCE = False
INCLUDE_INTERNAL_CMB_PRIORS = True

PLANCK_ROOT = Path('/Users/nguyenmn/candl/clipy/Planck_likelihoods/baseline/plc_3.0')
PLANCK_HIGHL = PLANCK_ROOT / 'hi_l/plik/plik_rd12_HM_v22b_TTTEEE.clik'
PLANCK_LOWL_TT = PLANCK_ROOT / 'low_l/commander/commander_dx12_v3_2_29.clik'
PLANCK_LOWL_EE = PLANCK_ROOT / 'low_l/simall/simall_100x143_offlike5_EE_Aplanck_B.clik'
PLANCK_LENSING = PLANCK_ROOT / 'lensing/smicadx12_Dec5_ftl_mv2_ndclpp_p_teb_consext8_CMBmarged.clik_lensing'
ACT_DR6_LENS = candl_data.ACT_DR6_Lens  # Index shortcut; upstream default resolves to lens_only. Use variant='use_CMB' or candl_data.ACT_DR6_Lens_and_CMB for the combined ACT lensing+CMB case.
EMULATOR_DIR = Path('/Users/nguyenmn/cosmopower-jax-for-pfs/cosmology/jense2024/jense_2023_camb_lcdm/networks')
CMB_EMULATOR_FILENAMES = {
    'TT': str(EMULATOR_DIR / 'jense_2023_camb_lcdm_Cl_tt.npz'),
    'TE': str(EMULATOR_DIR / 'jense_2023_camb_lcdm_Cl_te.npz'),
    'EE': str(EMULATOR_DIR / 'jense_2023_camb_lcdm_Cl_ee.npz'),
    'pp': str(EMULATOR_DIR / 'jense_2023_camb_lcdm_Cl_pp.npz'),
}
BAO_DATA_DIR = REPO_ROOT / 'ext_data/bao_data/desi_bao_dr2'
CACHE_DIR = get_cache_dir(REPO_ROOT)
ARTIFACT_PATH = artifact_path_for_selector(ACTIVE_COMBINATION, CACHE_DIR)

LAPTOP_MODE = True
if LAPTOP_MODE:
    NUM_WARMUP = 100
    NUM_SAMPLES = 800
    NUM_CHAINS = 2
    SCAN_CHUNK = 16
    MAX_TREE_DEPTH = (10, 10)
    PARALLEL_CHAINS = True
else:
    NUM_WARMUP = 200
    NUM_SAMPLES = 800
    NUM_CHAINS = 2
    SCAN_CHUNK = 128
    MAX_TREE_DEPTH = (10, 10)
    PARALLEL_CHAINS = False

MOSSABBN_MEAN = 0.02233
MOSSABBN_SIGMA = 0.00036
NS_PRIOR_MEAN = 0.96
NS_PRIOR_SIGMA = 0.02
COSMO_FALLBACK_SCALES = {
    '100theta': 0.005,
    'H0': 5.0,
    'ombh2': 3.6e-4,
    'omch2': 6.0e-3,
    'logA': 5.0e-2,
    'ns': 2.0e-2,
    'tau': 1.0e-2,
}
MANUAL_WHITENING_SCALES = {
    'ACT DR6 lensing + BBN': {
        'H0': 6.0,
        'ombh2': 4.5e-4,
        'omch2': 2.0e-2,
        'logA': 0.12,
        'ns': 0.03,
    },
    'Planck lensing + BAO + BBN': {
        '100theta': 0.005,
        'ombh2': 3.6e-4,
        'omch2': 6.0e-3,
        'logA': 5.0e-2,
        'ns': 2.0e-2,
    },
    'ACT + Planck lensing + BAO + BBN': {
        '100theta': 0.005,
        'ombh2': 3.6e-4,
        'omch2': 6.0e-3,
        'logA': 5.0e-2,
        'ns': 2.0e-2,
    },
}

ACTIVE_CONFIG = COMBINATION_CONFIGS[ACTIVE_COMBINATION]
print('Active selector:', ACTIVE_COMBINATION)
print('Label:', ACTIVE_CONFIG['label'])
print('CMB terms:', tuple(ACTIVE_CONFIG['cmb_term_names']))
print('Artifact path:', ARTIFACT_PATH)
print('Mode:', 'LAPTOP' if LAPTOP_MODE else 'SERVER')
print(f'  warmup={NUM_WARMUP}, samples={NUM_SAMPLES}, chains={NUM_CHAINS}')
print(f'  scan_chunk={SCAN_CHUNK}, max_tree_depth={MAX_TREE_DEPTH}, parallel_chains={PARALLEL_CHAINS}')
if ACTIVE_COMBINATION == 'planck_primary':
    print('Note: this run uses the local Planck 2018 plik + commander + simall stack, not the paper\'s exact PR4 + SRoll2 reference.')


Active selector: planck_primary
Label: Planck 2018 TTTEEE + lowT + lowE
CMB terms: ('planck_highl', 'planck_lowl_tt', 'planck_lowl_ee')
Artifact path: /Users/nguyenmn/jaxPTPolyPol/example/mcmc/cache/cmb_bao_bbn_lcdm/cmb_bao_bbn_LCDM_planck_primary.npz
Mode: LAPTOP
  warmup=100, samples=800, chains=2
  scan_chunk=16, max_tree_depth=(10, 10), parallel_chains=True
Note: this run uses the local Planck 2018 plik + commander + simall stack, not the paper's exact PR4 + SRoll2 reference.


In [3]:
FIDUCIAL_NATIVE = dict(DEFAULT_FIDUCIAL_NATIVE)
FIDUCIAL_100THETA = scalar_value(
    100.0 * bg.theta_star(
        FIDUCIAL_NATIVE['ombh2'],
        FIDUCIAL_NATIVE['omch2'],
        FIDUCIAL_NATIVE['H0'] / 100.0,
        mnu=MNU_FIXED,
        neff=NEFF_FIXED,
    )
)
FIDUCIAL_SAMPLED = {
    '100theta': FIDUCIAL_100THETA,
    'H0': FIDUCIAL_NATIVE['H0'],
    'ombh2': FIDUCIAL_NATIVE['ombh2'],
    'omch2': FIDUCIAL_NATIVE['omch2'],
    'logA': FIDUCIAL_NATIVE['logA'],
    'ns': FIDUCIAL_NATIVE['ns'],
    'tau': FIDUCIAL_NATIVE['tau'],
}


def load_likelihood_terms(term_names, include_internal_priors=True):
    clipy_args = {'all_priors': True} if include_internal_priors else {}
    loaded = {}
    if 'planck_highl' in term_names:
        loaded['planck_highl'] = load_candl_likelihood(
            str(PLANCK_HIGHL), wrapper='clipy', additional_args=clipy_args
        )
    if 'planck_lowl_tt' in term_names:
        loaded['planck_lowl_tt'] = load_candl_likelihood(
            str(PLANCK_LOWL_TT), wrapper='clipy', additional_args=clipy_args
        )
    if 'planck_lowl_ee' in term_names:
        loaded['planck_lowl_ee'] = load_candl_likelihood(
            str(PLANCK_LOWL_EE), wrapper='clipy', additional_args=clipy_args
        )
    if 'planck_lensing' in term_names:
        loaded['planck_lensing'] = load_candl_likelihood(
            str(PLANCK_LENSING), wrapper='clipy', additional_args=clipy_args
        )
    if 'act_dr6_lensing' in term_names:
        loaded['act_dr6_lensing'] = load_candl_likelihood(
            ACT_DR6_LENS,
            lensing=True,
            feedback=False,
            clear_internal_priors=not include_internal_priors,
        )
    return loaded


def make_lcdm_bao_loglike_fn(bao_data, *, mnu_fixed=MNU_FIXED):
    bao_native_fn = make_bao_theory_fn(
        bao_data,
        cosmo_keys=('ombh2', 'omch2', 'h'),
        cosmo_sizes=(1, 1, 1),
        mnu_fixed=mnu_fixed,
    )
    data = jnp.asarray(bao_data.data_vector, dtype=jnp.float64)
    cov_inv = jnp.asarray(inv(bao_data.cov), dtype=jnp.float64)

    @jax.jit
    def bao_loglike_native(native_theta):
        native_theta = jnp.asarray(native_theta, dtype=jnp.float64)
        H0, ombh2, omch2 = native_theta[0], native_theta[1], native_theta[2]
        bao_params = jnp.array([ombh2, omch2, H0 / 100.0], dtype=jnp.float64)
        theory = bao_native_fn(bao_params)
        residual = data - theory
        return -0.5 * residual @ cov_inv @ residual

    return bao_loglike_native


def nuisance_scale(default):
    default = float(default)
    return max(abs(default) * 0.1, 1.0e-3)


def combo_prior_logp(prior_mode, theta_cosmo, sampled_keys, native_theta):
    sampled = {key: theta_cosmo[i] for i, key in enumerate(sampled_keys)}
    H0 = native_theta[0]
    ombh2 = native_theta[1]
    omch2 = native_theta[2]
    logA = native_theta[3]
    ns = native_theta[4]
    tau = native_theta[5]

    inside_common = (
        (40.0 <= H0) & (H0 <= 100.0)
        & (0.005 <= omch2) & (omch2 <= 0.99)
        & (1.61 <= logA) & (logA <= 4.0)
    )
    if '100theta' in sampled_keys:
        theta100 = sampled['100theta']
        inside_common = inside_common & ((0.5 <= theta100) & (theta100 <= 10.0))

    if prior_mode == 'planck_aniso':
        inside = (
            inside_common
            & (0.005 <= ombh2) & (ombh2 <= 0.1)
            & (0.8 <= ns) & (ns <= 1.2)
            & (0.01 <= tau) & (tau <= 0.8)
        )
        return jnp.where(inside, 0.0, -jnp.inf)

    if prior_mode in {'lensing_bbn', 'bao_bbn'}:
        inside = inside_common & (0.85 <= ns) & (ns <= 1.10)
        gauss = -0.5 * ((ombh2 - MOSSABBN_MEAN) / MOSSABBN_SIGMA) ** 2
        gauss += -0.5 * ((ns - NS_PRIOR_MEAN) / NS_PRIOR_SIGMA) ** 2
        return jnp.where(inside, gauss, -jnp.inf)

    raise ValueError(f'unknown prior_mode {prior_mode!r}')


cmb_term_names = tuple(ACTIVE_CONFIG['cmb_term_names'])
sampled_cosmo_keys = tuple(ACTIVE_CONFIG['sampled_cosmo_keys'])
likelihoods = load_likelihood_terms(cmb_term_names, include_internal_priors=INCLUDE_INTERNAL_CMB_PRIORS)
pars_to_theory_specs = make_candl_pars_to_theory_specs_fn(emulator_filenames=CMB_EMULATOR_FILENAMES)

nuisance_names = ordered_union(
    get_candl_parameter_names(likelihoods[name], cosmo_keys=COSMO_KEYS_NATIVE, include_prior_params=True)
    for name in cmb_term_names
) if MARGINALIZE_CMB_NUISANCE else ()

nuisance_defaults = {}
for term_name, like in likelihoods.items():
    for key, value in get_candl_default_parameters(like).items():
        nuisance_defaults.setdefault(key, scalar_value(value))

layout = CandlParameterLayout(
    cosmo_keys=COSMO_KEYS_NATIVE,
    cosmo_sizes=COSMO_SIZES_NATIVE,
    cmb_nuisance_names=tuple(nuisance_names),
)
fixed_cmb_params = {
    name: nuisance_defaults[name]
    for name in ordered_union(get_candl_parameter_names(likelihoods[n], cosmo_keys=COSMO_KEYS_NATIVE, include_prior_params=True) for n in cmb_term_names)
    if name not in set(nuisance_names)
}

cmb_term_loglikes = {
    term_name: make_candl_loglike_fn(
        likelihoods[term_name],
        pars_to_theory_specs=pars_to_theory_specs,
        layout=layout,
        fixed_cmb_params=fixed_cmb_params,
        jit_compile=True,
    )
    for term_name in cmb_term_names
}

bao_loglike_native = None
bao_data = None
if ACTIVE_CONFIG['include_bao']:
    bao_data = load_desi_dr2('all', data_dir=BAO_DATA_DIR)
    bao_loglike_native = make_lcdm_bao_loglike_fn(bao_data, mnu_fixed=MNU_FIXED)

theta_fid_cosmo = jnp.asarray([FIDUCIAL_SAMPLED[key] for key in sampled_cosmo_keys], dtype=jnp.float64)
theta_fid_nuis = jnp.asarray([nuisance_defaults[name] for name in nuisance_names], dtype=jnp.float64)
theta_fid = jnp.concatenate([theta_fid_cosmo, theta_fid_nuis]) if theta_fid_nuis.size else theta_fid_cosmo
fallback_scales = jnp.asarray(
    [COSMO_FALLBACK_SCALES[key] for key in sampled_cosmo_keys]
    + [nuisance_scale(nuisance_defaults[name]) for name in nuisance_names],
    dtype=jnp.float64,
)


def sampled_to_native_and_layout(theta_varied):
    theta_varied = jnp.asarray(theta_varied, dtype=jnp.float64)
    n_cosmo = len(sampled_cosmo_keys)
    theta_cosmo = theta_varied[:n_cosmo]
    theta_nuis = theta_varied[n_cosmo:]
    native_dict = native_cosmo_dict_from_sampled(
        theta_cosmo,
        sampled_cosmo_keys,
        fiducial_native=FIDUCIAL_NATIVE,
        fiducial_sampled=FIDUCIAL_SAMPLED,
        mnu_fixed=MNU_FIXED,
        neff_fixed=NEFF_FIXED,
    )
    native_cosmo = CosmoParams(native_dict)
    nuisance_map = {name: theta_nuis[i] for i, name in enumerate(nuisance_names)}
    layout_theta = layout.pack(native_cosmo, nuisance_map)
    return theta_cosmo, native_cosmo.to_array(), layout_theta


@jax.jit
def log_post(theta_varied):
    theta_cosmo, native_theta, layout_theta = sampled_to_native_and_layout(theta_varied)
    logp = combo_prior_logp(ACTIVE_CONFIG['prior_mode'], theta_cosmo, sampled_cosmo_keys, native_theta)
    if bao_loglike_native is not None:
        logp = logp + bao_loglike_native(native_theta)
    for fn in cmb_term_loglikes.values():
        logp = logp + fn(layout_theta)
    return logp

fid_theta_cosmo, fid_native, fid_layout = sampled_to_native_and_layout(theta_fid)
fid_terms = {}
if bao_loglike_native is not None:
    fid_terms['bao'] = scalar_value(bao_loglike_native(fid_native))
for term_name, fn in cmb_term_loglikes.items():
    fid_terms[term_name] = scalar_value(fn(fid_layout))

spec = {
    'selector': ACTIVE_COMBINATION,
    'label': ACTIVE_CONFIG['label'],
    'sampled_cosmo_keys': sampled_cosmo_keys,
    'cmb_term_names': cmb_term_names,
    'cmb_nuisance_names': tuple(nuisance_names),
    'theta_fid': theta_fid,
    'fallback_scales': fallback_scales,
    'log_post': log_post,
    'fid_native': fid_native,
    'fid_terms': fid_terms,
    'include_bao': ACTIVE_CONFIG['include_bao'],
    'prior_mode': ACTIVE_CONFIG['prior_mode'],
}

print('Fiducial log posterior:', scalar_value(spec['log_post'](spec['theta_fid'])))
for key, value in fid_terms.items():
    print(f'{key:>20s}: {value:.6f}')
if ACTIVE_COMBINATION == 'planck_primary':
    print('Planck primary components loaded:', ', '.join(cmb_term_names))


----
clipy_0.15
Checking likelihood '/Users/nguyenmn/candl/clipy/Planck_likelihoods/baseline/plc_3.0/hi_l/plik/plik_rd12_HM_v22b_TTTEEE.clik' on test data. got -1172.47 expected -1172.47 (diff 4.00569e-06)
----
----
clipy_0.15
Checking likelihood '/Users/nguyenmn/candl/clipy/Planck_likelihoods/baseline/plc_3.0/low_l/commander/commander_dx12_v3_2_29.clik' on test data. got -11.6257 expected -11.6257 (diff 1.02746e-06)
----
----
clipy_0.15
Checking likelihood '/Users/nguyenmn/candl/clipy/Planck_likelihoods/baseline/plc_3.0/low_l/simall/simall_100x143_offlike5_EE_Aplanck_B.clik' on test data. got -197.99 expected -197.99 (diff -4.1778e-08)
----
Tried to load pickle file from pre-trained model, but failed.
This usually means that you have TF>=2.14, or that you are loading a model that was trained on PCA but loaded with the log (or viceversa), or that you are loading a non-standard model from the cosmopower-organization repo.
Falling back to the dictionary, in case this also fails or does n

In [4]:
def whitening_scales_from_hessian(log_post_physical, theta_fid, fallback_scales):
    nll = lambda theta: -log_post_physical(theta)
    hess = np.asarray(jax.jit(jax.hessian(nll))(theta_fid), dtype=float)
    fisher = 0.5 * (hess + hess.T)
    cov = np.linalg.pinv(fisher)
    scales = np.sqrt(np.clip(np.diag(cov), 0.0, np.inf))
    fallback = np.asarray(fallback_scales, dtype=float)
    bad = (~np.isfinite(scales)) | (scales <= 0.0)
    scales[bad] = fallback[bad]
    scales = np.maximum(scales, 0.20 * fallback)
    return fisher, cov, scales


def manual_scales_for_spec(spec):
    entries = MANUAL_WHITENING_SCALES.get(spec['label'])
    if not entries:
        return None
    return jnp.asarray([entries[key] for key in spec['sampled_cosmo_keys']], dtype=jnp.float64)


def run_combination_chain(spec, *, seed):
    manual_scales = manual_scales_for_spec(spec)
    if manual_scales is None:
        fisher_local, cov_local, scales = whitening_scales_from_hessian(
            spec['log_post'], spec['theta_fid'], spec['fallback_scales']
        )
    else:
        fisher_local = None
        cov_local = None
        scales = manual_scales
    to_whitened, to_physical = make_transform(center=spec['theta_fid'], scale=scales)

    @jax.jit
    def log_post_whitened(x):
        return spec['log_post'](to_physical(x))

    x0 = jnp.zeros_like(spec['theta_fid'])
    samples_w, diagnostics = run_nuts(
        jax.random.key(seed),
        log_post_whitened,
        initial_position=x0,
        num_warmup=NUM_WARMUP,
        num_samples=NUM_SAMPLES,
        num_chains=NUM_CHAINS,
        adapt_mass_matrix=False,
        mass_matrix_type='diagonal',
        initial_inverse_mass_matrix=jnp.ones(spec['theta_fid'].shape[0]),
        max_tree_depth=MAX_TREE_DEPTH,
        scan_chunk_size=SCAN_CHUNK,
        parallel_chains=PARALLEL_CHAINS,
    )
    samples_phys = samples_to_physical(samples_w, to_physical)
    flat = np.asarray(samples_phys).reshape(-1, spec['theta_fid'].shape[0])
    flat_log_post = np.asarray(jax.vmap(spec['log_post'])(samples_phys.reshape(-1, spec['theta_fid'].shape[0])))
    flat_divergent = np.asarray(diagnostics['is_divergent']).reshape(-1).astype(bool)
    valid = ~flat_divergent
    flat_valid = flat[valid] if np.any(valid) else flat
    flat_log_post_valid = flat_log_post[valid] if np.any(valid) else flat_log_post
    map_idx = int(np.argmax(flat_log_post_valid))
    return {
        'spec': spec,
        'samples_w': np.asarray(samples_w),
        'samples_phys': np.asarray(samples_phys),
        'flat_samples': flat_valid,
        'flat_log_post': flat_log_post_valid,
        'diagnostics': diagnostics,
        'map_theta': flat_valid[map_idx],
        'whitening_scales': scales,
        'fisher_local': fisher_local,
        'cov_local': cov_local,
    }


result = run_combination_chain(spec, seed=SEED)
metadata = {
    'selector': spec['selector'],
    'label': spec['label'],
    'sampled_cosmo_keys': spec['sampled_cosmo_keys'],
    'cmb_term_names': spec['cmb_term_names'],
    'cmb_nuisance_names': spec['cmb_nuisance_names'],
    'prior_mode': spec['prior_mode'],
    'include_bao': spec['include_bao'],
    'settings': {
        'num_warmup': NUM_WARMUP,
        'num_samples': NUM_SAMPLES,
        'num_chains': NUM_CHAINS,
        'scan_chunk': SCAN_CHUNK,
        'max_tree_depth': list(MAX_TREE_DEPTH),
        'parallel_chains': PARALLEL_CHAINS,
        'seed': SEED,
    },
    'paths': {
        'planck_highl': str(PLANCK_HIGHL),
        'planck_lowl_tt': str(PLANCK_LOWL_TT),
        'planck_lowl_ee': str(PLANCK_LOWL_EE),
        'planck_lensing': str(PLANCK_LENSING),
        'act_dr6_lens': str(ACT_DR6_LENS),
        'bao_data_dir': str(BAO_DATA_DIR),
        'cmb_emulators': CMB_EMULATOR_FILENAMES,
    },
}

save_run_artifact(
    ARTIFACT_PATH,
    metadata=metadata,
    flat_samples=result['flat_samples'],
    flat_log_post=result['flat_log_post'],
    whitening_scales=result['whitening_scales'],
    fid_native=spec['fid_native'],
    acceptance_rate=np.asarray(result['diagnostics']['acceptance_rate']),
    num_integration_steps=np.asarray(result['diagnostics']['num_integration_steps']),
    is_divergent=np.asarray(result['diagnostics']['is_divergent']),
)
print('Saved artifact to', ARTIFACT_PATH)


Saved artifact to /Users/nguyenmn/jaxPTPolyPol/example/mcmc/cache/cmb_bao_bbn_lcdm/cmb_bao_bbn_LCDM_planck_primary.npz


In [5]:
accept = np.asarray(result['diagnostics']['acceptance_rate'])
steps = np.asarray(result['diagnostics']['num_integration_steps'])
divergent = np.asarray(result['diagnostics']['is_divergent'])

display(pd.DataFrame([{
    'dataset': spec['label'],
    'accept_mean': float(accept.mean()),
    'steps_mean': float(steps.mean()),
    'divergent_total': int(divergent.sum()),
    'saved_file': str(ARTIFACT_PATH.name),
}]))

print('MAP log posterior:', float(np.max(result['flat_log_post'])))
print('Saved samples shape:', result['flat_samples'].shape)


,dataset,accept_mean,steps_mean,divergent_total,saved_file
0,Planck 2018 TTTEEE + lowT + lowE,0.84941,996.32,0,cmb_bao_bbn_LCDM_planck_primary.npz


MAP log posterior: -1383.4175262908561
Saved samples shape: (1600, 6)
